In [1]:
# System related and data input controls
import os

# Ignore the warnings
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

# Library
import pandas as pd
import numpy as np
import math

# Visualization|
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# 한글 설정
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'

In [4]:
# Data Preprocessing
folder_location = os.path.join(os.path.join('..', 'data'))
Y_colname = '승차인원수'
RANDOM_STATE = 123
DATE_SPLITS = ['2023-03-31', '2024-03-31']

In [5]:
print(folder_location)

..\data


In [21]:
df = pd.read_csv(os.path.join(folder_location, 'df_KTX_monthsum_KK.csv'), encoding='utf-8-sig')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

,전체주중주말,주운행선,Date,공급차량수,공급좌석합계수,승차인원수,1인당단가,1인당거리,1좌석당단가,좌석회전율,...,관광,일반,대수송,임시,확정,시발역,종착역,시발종착역,열차운행횟수,1열차당승차인원
0,주말,경부선,2015-01-01,34020,1741181,1871129,496684.493797,37822.859888,533787.092457,15.054794,...,0,218,0,6,212,84,84,154,1991,13164.713829
1,주말,경부선,2015-02-01,28992,1488544,1533571,434612.946125,32762.087399,447295.799500,12.361342,...,0,188,46,6,136,72,72,132,1705,10793.213241
2,주말,경부선,2015-03-01,31246,1607627,1780862,454810.916585,34157.906870,503277.605271,14.399538,...,0,196,0,0,196,78,78,143,1837,12604.742509
3,주말,경부선,2015-04-01,28848,1491483,1681821,424444.354602,31774.297786,478264.495853,13.528454,...,0,195,0,0,195,72,72,132,1656,12186.130338
4,주말,경부선,2015-05-01,36102,1863967,2144330,531982.093900,40218.497239,611756.680000,17.251379,...,0,250,0,4,246,90,90,165,2072,15522.618462


In [22]:
# Excel 파일을 읽을 때는 이 함수를 사용하세요.
news_df = pd.read_excel(os.path.join(folder_location, 'NewsResult_20150101-20250331_KTX코레일_KK.xlsx'))
news_df.head(2)

,뉴스 식별자,일자,언론사,기고자,제목,통합 분류1,통합 분류2,통합 분류3,사건/사고 분류1,사건/사고 분류2,사건/사고 분류3,인물,위치,기관,키워드,특성추출(가중치순 상위 50개),본문,URL,분석제외 여부
0,2.100501e+06,20250331,파이낸셜뉴스,성석우 기자 (west@fnnews.com),11억명 실어나른 1세대 KTX 교체 골든타임 2년 남았다,지역>강원,지역>대전,경제>자원,NaN,NaN,NaN,NaN,"영국,지구","미국,정부,한국철도공사,아일랜드,KT,코레일,국회,AMTRAK,KTX","11억,KTX,2년,교체,골든타임,개통,21주년,지구,둘레,바퀴,거리,운행,차세대,...","코레일,ktx,차세대,고속철,주년,고속열차,1만,이용객,운영사,시운전,달해",2004년 4월 개통 올해 21주년 \n지구둘레 1만7000바퀴 거리 운행 \n차세...,http://www.fnnews.com/news/202503311817021622,NaN
1,1.100701e+06,20250329,세계일보,김동환,"옅은 냄새에 색깔은 선명, 광택도 번쩍 KTX ‘친환경 페인트’ 도색 가보니",지역>강원,사회>사회일반,지역>대전,사고>산업사고>화재,NaN,NaN,NaN,"수도권철도차량정비단,서울,고양,경기,하도,부산,고양시,수도권철도,수도권","수성,한국철도공사,KT,코레일,KCC","옅은,냄새,색깔,선명,광택,페인트,KTX,도색,경기,고양시,수도,철도차량정비단,중정...","ktx,코레일,김동환,유성,만큼",지난 27일 경기 고양시 수도권철도차량정비단 중정비동 내에서 외부 도색 작업을 마친...,http://www.segye.com/content/html/2025/03/28/2...,NaN


In [26]:
news_df['사건/사고 분류1'].unique()

array([nan, ' 사고>교통사고>철도사고', ' 사고>산업사고>화재', ' 사고>스포츠사고', ' 사고>교통사고>항공사고',
       ' 사회>사회갈등>전쟁', ' 사회>사회갈등>시위', ' 사고>교통사고>노상사고', ' 사회>사회갈등>테러행위',
       ' 재해>자연재해>눈사태_산사태', ' 재해>자연재해>태풍', ' 재해>자연재해>폭염', ' 재해>자연재해>지진',
       ' 재해>자연재해>홍수', ' 범죄>성범죄>성추행', ' 사고>교통사고>해상사고', ' 범죄>범죄일반>폭행',
       ' 범죄>성범죄>성희롱', ' 사고>산업사고>폭발', ' 범죄>범죄일반>살인', ' 범죄>범죄일반>방화',
       ' 사회>사회문제>노예', ' 사고>산업사고>붕괴', ' 사회>사회문제>학대', ' 범죄>범죄일반>절도',
       ' 사회>사회문제>미성년범죄', ' 사회>사회갈등>반란_혁명_폭동', ' 범죄>기업범죄>횡령',
       ' 범죄>기업범죄>내부자거래', ' 범죄>범죄일반>유괴/납치', ' 사회>사회문제>성차별', ' 범죄>범죄일반>사기',
       ' 재해>자연재해>미세먼지_황사', ' 범죄>정치>뇌물수수', ' 범죄>성범죄>음란물', ' 범죄>기업범죄>거래제한',
       ' 범죄>범죄일반>마약'], dtype=object)

In [27]:
news_df['사건/사고 분류2'].unique()

array([nan, ' 사고>교통사고>철도사고', ' 사고>산업사고>화재', ' 사고>스포츠사고', ' 사고>교통사고>항공사고',
       ' 사회>사회갈등>전쟁', ' 사회>사회갈등>시위', ' 사고>교통사고>노상사고', ' 사회>사회갈등>테러행위',
       ' 재해>자연재해>눈사태_산사태', ' 재해>자연재해>태풍', ' 재해>자연재해>폭염', ' 재해>자연재해>지진',
       ' 재해>자연재해>홍수', ' 범죄>성범죄>성추행', ' 사고>교통사고>해상사고', ' 범죄>범죄일반>폭행',
       ' 범죄>성범죄>성희롱', ' 사고>산업사고>폭발', ' 범죄>범죄일반>살인', ' 범죄>범죄일반>방화',
       ' 사회>사회문제>노예', ' 사고>산업사고>붕괴', ' 사회>사회문제>학대', ' 범죄>범죄일반>절도',
       ' 사회>사회문제>미성년범죄', ' 사회>사회갈등>반란_혁명_폭동', ' 범죄>기업범죄>횡령',
       ' 범죄>기업범죄>내부자거래', ' 범죄>범죄일반>유괴/납치', ' 사회>사회문제>성차별', ' 범죄>범죄일반>사기',
       ' 재해>자연재해>미세먼지_황사', ' 범죄>정치>뇌물수수', ' 범죄>성범죄>음란물', ' 범죄>기업범죄>거래제한',
       ' 범죄>범죄일반>마약'], dtype=object)

In [28]:
news_df['사건/사고 분류3'].unique()

array([nan, ' 사회>사회갈등>전쟁', ' 사고>교통사고>철도사고', ' 사고>교통사고>해상사고', ' 사고>스포츠사고',
       ' 사회>사회갈등>시위', ' 재해>자연재해>폭염', ' 재해>자연재해>태풍', ' 재해>자연재해>지진',
       ' 재해>자연재해>눈사태_산사태', ' 재해>자연재해>홍수', ' 사고>교통사고>노상사고', ' 범죄>범죄일반>살인',
       ' 사회>사회문제>성차별', ' 사고>산업사고>화재', ' 범죄>범죄일반>사기', ' 사고>산업사고>붕괴',
       ' 사회>사회문제>학대', ' 사고>교통사고>항공사고', ' 범죄>범죄일반>방화', ' 사회>사회갈등>반란_혁명_폭동',
       ' 범죄>범죄일반>절도', ' 범죄>범죄일반>폭행', ' 범죄>성범죄>성희롱'], dtype=object)

In [29]:
news_df['통합 분류1'].unique()

array(['지역>강원', '경제>유통', '사회>사회일반', '정치>정치일반', '경제>자원', '경제>부동산',
       '경제>서비스_쇼핑', '경제>경제일반', '지역>대전', '경제>산업_기업', '경제>금융_재테크',
       '사회>교육_시험', '문화>요리_여행', '지역>대구', '사회>여성', 'IT_과학>모바일', '지역>부산',
       '지역>광주', '정치>행정_자치', '문화>종교', '문화>학술_문화재', '지역>경북', '사회>장애인',
       '지역>경기', '사회>사건_사고', 'IT_과학>IT_과학일반', 'IT_과학>인터넷_SNS', '경제>자동차',
       '문화>미술_건축', '사회>미디어', '사회>날씨', '지역>전북', '지역>충남', '지역>제주', '문화>생활',
       '문화>문화일반', '지역>전남', '경제>국제경제', '문화>출판', '경제>무역', '지역>경남',
       '문화>방송_연예', '지역>울산', '지역>지역일반', '문화>전시_공연', 'IT_과학>콘텐츠',
       '사회>노동_복지', '국제>아시아', '정치>청와대', '경제>취업_창업', '정치>국회_정당', 'IT_과학>보안',
       '문화>영화', '사회>의료_건강', '경제>증권_증시', '정치>외교', '경제>반도체', '문화>음악',
       '스포츠>골프', '국제>유럽_EU', '경제>외환', '지역>충북', '정치>선거', 'IT_과학>과학',
       '정치>북한', '국제>미국_북미', '국제>러시아', '스포츠>스포츠일반', '사회>환경', '국제>일본',
       '국제>국제일반', '국제>중국', '국제>중남미'], dtype=object)

In [24]:
pd.DataFrame({
    'Null_Count': news_df.isnull().sum(),
    'Unique_Count': news_df.nunique()
}).sort_values(by=['Unique_Count', 'Null_Count'], ascending=False)

,Null_Count,Unique_Count
키워드,0,10899
본문,0,10890
특성추출(가중치순 상위 50개),0,10865
제목,0,10664
URL,580,10425
기관,4,8854
뉴스 식별자,0,8823
위치,914,7657
기고자,1031,3080
인물,5366,2702
